# NOTE: reference-only notebook (does not run in this fork)

**This notebook is kept as documentation, not as runnable code.** It comes from the model-*training* workflow, which has been archived in this research fork -- only the inference pipeline is live here. The notebook is preserved to document how cloud masking affected the training imagery, in support of the planned "cloud flag" research.

**Why it fails to run:** the import `from scripts import viz_tools` breaks, because `viz_tools.py` now lives in `_archive/deprecated/`. To actually execute the notebook, restore `_archive/deprecated/viz_tools.py` (e.g., copy it back into a `scripts/` package on the import path). The `WORK_DIR` training-patch snapshot (`../data/training_patches2025-09-26T15:38/`) is also not shipped in this fork.

**What it did in the training workflow:** training patches are cut from cloud-masked Sentinel-2 composites; wherever every scene in the date window was cloudy (Cloud Score+ `cs_cdf` below the clear threshold), the masked pixels come through as (near-)black. This notebook finds patches containing such masked pixels, removes them from the **positive** class (a partly-masked "mine" chip is a corrupted label), and -- deliberately -- keeps them in the **negative** class, so the classifier learns that masked/cloudy pixels are not evidence of mining.

### Optional: consider removing cloud-masked training samples, especially from the positive training dataset

A few partly-masked images in the negative dataset is probably healthy

In [ ]:
# ============================================================
# Cell 1 -- Imports and paths (BROKEN IN THIS FORK -- see note above)
# ============================================================
# --- Standard library ---
# glob: discover training-patch .tif files; shutil: move rejected patches
# into a quarantine folder; random: sample patches for the visual grids;
# date/os/sys: misc path and environment plumbing.
from datetime import date
import glob
import os
import random
import shutil
import sys

# --- Third party ---
# matplotlib: titles/figures for the image grids; numpy: patch arrays;
# rasterio: reads the GeoTIFF training patches.
import matplotlib.pyplot as plt
import numpy as np
import rasterio

# Put the repo root (parent of this gee/ directory) on sys.path so that
# "from scripts import ..." can resolve.
parent_dir = os.path.split(os.getcwd())[0]
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# !! THIS IMPORT FAILS IN THIS FORK: viz_tools.py was moved to
# _archive/deprecated/ when the training code was archived. Restore
# _archive/deprecated/viz_tools.py into a scripts/ package (or adjust this
# import) to run the notebook. It provides plot_numpy_grid, used below to
# tile many patches into a single figure.
from scripts import viz_tools

# Root of a timestamped training-patch snapshot (not shipped in this fork).
# Layout: train/1/*.tif = positive patches (mines), train/0/*.tif =
# negative patches (background).
WORK_DIR = '../data/training_patches2025-09-26T15:38/'

# Auto-reload edited local modules while iterating.
%load_ext autoreload
%autoreload 2

In [ ]:
# ============================================================
# Cell 2 -- Load all training patches into class-wise arrays
# ============================================================
def read_tif(path):
    """Read a GeoTIFF patch into a float32 (H, W, bands) array.

    Args:
        path: path to a single training-patch GeoTIFF.

    Returns:
        float32 array of shape (H, W, C). rasterio reads (bands, H, W);
        np.moveaxis converts to channels-last. Values are Sentinel-2
        digital numbers (reflectance x 10000), left unscaled here.
    """
    with rasterio.open(path) as f:
        img = f.read()
    img = np.moveaxis(img, 0, -1)
    return img.astype('float32')

# Discover and stack every patch per class: np.array over N patches of
# shape (H, W, C) gives a (N, H, W, C) float32 array (all patches share
# one size). Directory name is the label: 1 = mine, 0 = background.
positive_paths = glob.glob(f"{WORK_DIR}/train/1/*.tif")
positive_patches = np.array([read_tif(p) for p in positive_paths])
negative_paths = glob.glob(f"{WORK_DIR}/train/0/*.tif")
negative_patches = np.array([read_tif(p) for p in negative_paths])

# Quick census of the two classes.
print(len(positive_patches), "positive samples loaded")
print(len(negative_patches), "negative samples loaded")

In [ ]:
# ============================================================
# Cell 3 -- Detector for cloud-masked ("black") pixels
# ============================================================
# Cloud-masked composite pixels had no clear-sky observations in the whole
# date window, so they come through as (near-)zero digital numbers.
# Darkness in a reflective band is therefore a proxy for "was masked".
def filter_black(data, mask_limit=0.1, dark_level=10):
    """Split patches by their fraction of near-black (masked) pixels.

    Args:
        data: (N, H, W, C) float32 patch array (S2 digital numbers).
        mask_limit: maximum tolerated fraction of dark pixels per patch
            (0.1 = 10%; 0 means reject a patch with ANY dark pixel).
        dark_level: darkness cutoff in digital numbers (10 DN = 0.001
            reflectance -- essentially only masked/empty pixels qualify).

    Returns:
        (filtered_data, rejected_data, rejected_indices): patches passing
        the test, patches failing it, and the failing indices into `data`
        (used later to move the corresponding files on disk).

    Note: patch[:, :, 1:2] selects only channel index 1 (band B2, blue),
    so the mean over the last axis is a no-op; the test is effectively
    "fraction of blue-band pixels darker than dark_level".
    """
    # Per-patch dark fraction: count pixels below dark_level in the selected
    # channel, divide by the pixel count -> (N,) float array in [0, 1].
    masked_fraction = np.array([np.sum(np.mean(patch[:,:,1:2], axis=-1) < dark_level) / np.size(np.mean(patch[:,:,1:2], axis=-1)) for patch in data])
    # Keep patches at or below the tolerated dark fraction.
    filtered_data = data[masked_fraction <= mask_limit]
    # Report the pass rate as a percentage.
    print(f"{len(filtered_data) / len(data) :.4%} of data passes filter")

    # Indices of the failing patches, for the file moves in later cells.
    rejected_indices = np.where(masked_fraction > mask_limit)[0]
    rejected_data = data[rejected_indices]
    return filtered_data, rejected_data, rejected_indices


### Positive class: zero tolerance for masked pixels
Scan every positive patch, visualize the rejects, then move their files out of the training set.

In [ ]:
# ============================================================
# Cell 4 -- Reject ANY masked positives, and eyeball the rejects
# ============================================================
# mask_limit=0: a positive (mine) patch with even one masked pixel is
# rejected. Rationale: a partly-masked "mine" chip is a corrupted positive
# label, and the positive set is small enough that purity beats volume.
# Filter pixels that are masked beyond a threshold. We want to avoid giving positive examples of cloud-masked patches
filtered, rejects, rejected_indices = filter_black(positive_patches, mask_limit = 0)

# Visual audit of what is being discarded (up to 256 patches).
if len(rejects) > 0:
    # Cap the grid at 256 patches.
    num_samples = min(len(rejects), 256)
    # Channels 3:0:-1 select band indices (3, 2, 1) = (B4, B3, B2) =
    # (red, green, blue); /3000 maps ~0.3 reflectance to 1.0 for display.
    rgbs = rejects[:,:,:,3:0:-1] / 3000
    # Random subset of the RGB patches for the grid figure.
    viz_samples = random.sample(list(rgbs), num_samples)
    # plot_numpy_grid tiles the patches into one matplotlib figure.
    viz_tools.plot_numpy_grid(viz_samples)
    plt.title("Masked Rejects")

In [ ]:
# ============================================================
# Cell 5 -- Quarantine the rejected positive files on disk
# ============================================================
# SIDE EFFECT: moves each rejected positive .tif from train/1/ into a
# train/1/cloud_mask_rejects/ subfolder. The training loader globs
# train/1/*.tif, so moved files silently drop out of the training set
# while remaining on disk for inspection or recovery.
for src in np.array(positive_paths)[rejected_indices]:
    dest_dir = os.path.join(os.path.dirname(src), 'cloud_mask_rejects')
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, os.path.basename(src))
    shutil.move(src, dest)

### Negative class: inspect, but (mostly) keep
Run the same scan on negatives for inspection; the removal cell at the end is left as a no-op (`n_to_remove = 0`).

In [ ]:
# ============================================================
# Cell 6 -- Same scan for the NEGATIVE class (inspection only)
# ============================================================
# Unlike positives, masked negatives are NOT all removed: a cloudy or
# partly-masked background patch is still a legitimate negative, and
# keeping some deliberately teaches the classifier that mask artifacts and
# clouds are not evidence of mining (see the note at the top of the
# notebook: a few partly-masked negatives are "probably healthy").
# Filter pixels that are masked beyond a threshold. 
filtered, rejects, rejected_indices = filter_black(negative_patches, mask_limit = 0)

# Visual audit of the masked negatives (up to 256 patches).
if len(rejects) > 0:
    num_samples = min(len(rejects), 256)
    # Same RGB rendering as Cell 4: bands (B4, B3, B2), /3000 for display.
    rgbs = rejects[:,:,:,3:0:-1] / 3000
    viz_samples = random.sample(list(rgbs), num_samples)
    viz_tools.plot_numpy_grid(viz_samples)
    plt.title("Masked Rejects")

In [ ]:
# ============================================================
# Cell 7 -- Sanity-check the kept (filtered) negatives
# ============================================================
# Render 256 random surviving negatives as a 16x16 RGB grid (bands
# B4/B3/B2, /3000 display scaling) to confirm the filter kept clean,
# usable imagery. NOTE: `filtered` still refers to the NEGATIVE-class
# result from Cell 6 (the variables are reused between cells).
num_samples = 16 ** 2
indices = np.random.randint(0, len(filtered), num_samples)
fig = viz_tools.plot_numpy_grid(filtered[indices,:,:,3:0:-1] / 3000)
plt.title('Filtered Samples')
plt.show()

In [ ]:
# ============================================================
# Cell 8 -- Collect and shuffle the cloudy negative file paths
# ============================================================
# File paths of the negatives flagged as masked/cloudy in Cell 6. Shuffled
# so that any partial removal in the next cell takes an unbiased random
# sample rather than a glob-ordered one.
cloudy = np.array(negative_paths)[rejected_indices]
random.shuffle(cloudy)
print(len(cloudy))

In [ ]:
# ============================================================
# Cell 9 -- (Deliberate no-op) optionally thin the cloudy negatives
# ============================================================
# With n_to_remove = 0, cloudy[:0] is an empty slice and NOTHING is moved:
# the recorded decision was to keep all cloudy negatives in training.
# Raising n_to_remove would quarantine that many randomly-chosen cloudy
# negatives into train/0/cloud_mask_rejects/ (same pattern as Cell 5).
n_to_remove = 0
for src in cloudy[:n_to_remove]:
    dest_dir = os.path.join(os.path.dirname(src), 'cloud_mask_rejects')
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, os.path.basename(src))
    shutil.move(src, dest)